# ImagineAI: warm-started LCM-LoRA distillation on CV-curated COCO (Colab)
Trains our few-step adapter on a free Colab GPU instead of the laptop. **Runtime -> Change runtime type -> T4 GPU.**
Before running: upload the three files in `colab_upload/ImagineAI_LCM/` (`coco_train.tar`, `coco_train.manifest.json`, `lcm_code_pack.zip`) to Google Drive at `MyDrive/ImagineAI_LCM/`.
Checkpoints are written to Drive every 250 steps; if the runtime disconnects just rerun all cells and training resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,1), 'GiB')
D = Path('/content/drive/MyDrive/ImagineAI_LCM')
for f in ['coco_train.tar','coco_train.manifest.json','lcm_code_pack.zip']:
    assert (D/f).is_file(), f'missing {D/f}'

In [ ]:
%pip install -q "diffusers==0.35.1" "transformers<5" "peft>=0.14" "accelerate>=0.30" "webdataset==1.0.2" "braceexpand==0.1.7" safetensors

In [ ]:
import hashlib, json, shutil, zipfile
meta = json.loads((D/'coco_train.manifest.json').read_text())
h = hashlib.sha256()
with (D/'coco_train.tar').open('rb') as f:
    for c in iter(lambda: f.read(1<<20), b''): h.update(c)
assert h.hexdigest() == meta['archive_sha256'], 'archive hash mismatch'
assert not meta['contains_validation_split']
shutil.copy2(D/'coco_train.tar', '/content/coco_train.tar')
zipfile.ZipFile(D/'lcm_code_pack.zip').extractall('/content/code')
import hashlib as H
t = Path('/content/code/ai_engine/training/third_party/train_lcm_distill_lora_sd_wds.py').read_bytes()
assert H.sha256(t).hexdigest() == '91574a77355539f44cacb229596eeb97e59da20ab969b2fe868132aaeb1047ec', 'trainer is not the pinned upstream file'
print('ok:', meta['training_pairs'], 'pairs; trainer hash verified')

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
D = Path('/content/drive/MyDrive/ImagineAI_LCM')
# Train. alpha MUST be 8: the public LCM-LoRA is rank 64 / alpha 8 (scale 0.125). Wrong alpha = garbage images.
# bs 4 x accum 2 = effective batch 8 (the laptop run used 4). Resumes from the latest Drive checkpoint automatically.
OUT = '/content/drive/MyDrive/ImagineAI_LCM/run_v2'
resume = ['--resume_from_checkpoint=latest'] if Path(OUT).exists() and any(Path(OUT).glob('checkpoint-*')) else []
cmd = ['accelerate','launch','--num_processes=1','--mixed_precision=fp16','/content/code/ai_engine/training/lcm_warmstart.py',
  '--pretrained_teacher_model=stable-diffusion-v1-5/stable-diffusion-v1-5','--teacher_revision=451f4fe16113bff5a5d2269ed5ad43b0592e9a14',
  '--init_lora_from=latent-consistency/lcm-lora-sdv1-5', f'--output_dir={OUT}',
  '--train_shards_path_or_url=/content/coco_train.tar','--resolution=512','--train_batch_size=4','--gradient_accumulation_steps=2',
  '--max_train_steps=2000','--max_train_samples=7000','--learning_rate=1e-5','--lr_scheduler=cosine','--lr_warmup_steps=50',
  '--lora_rank=64','--lora_alpha=8','--mixed_precision=fp16','--cast_teacher_unet','--gradient_checkpointing',
  '--vae_encode_batch_size=4','--dataloader_num_workers=2','--checkpointing_steps=250','--checkpoints_total_limit=8',
  '--validation_steps=1000000','--report_to=tensorboard','--seed=2026'] + resume
env = dict(os.environ, PYTHONPATH='/content/code')
LOG = '/content/train.log'
with open(LOG, 'w') as lf:
    r = subprocess.run(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
lines = Path(LOG).read_text(errors='replace').splitlines()
print(chr(10).join(lines[-40:]))   # the real error (or training progress) is here
if r.returncode != 0:
    shutil.copy2(LOG, '/content/drive/MyDrive/ImagineAI_LCM/last_train_error.log')
    raise SystemExit(f'Training failed (exit {r.returncode}). Copy the lines above ("Error"/"Traceback") and send them.')

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
D = Path('/content/drive/MyDrive/ImagineAI_LCM')
# Export every checkpoint to a diffusers-loadable adapter (B matrices scaled by alpha/rank = 8/64).
import sys; sys.path.insert(0,'/content/code')
from ai_engine.training.export_lcm_adapter import export
OUT = '/content/drive/MyDrive/ImagineAI_LCM/run_v2'
EXP = D/'exports'; EXP.mkdir(exist_ok=True)
for ck in sorted(Path(OUT).glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1])):
    n = ck.name.split('-')[1]
    print(n, export(ck/'unet_lora', EXP/f'lcm_curated_{n}', alpha=8, rank=64)['n_tensors'], 'tensors')
if (Path(OUT)/'unet_lora').exists():
    export(Path(OUT)/'unet_lora', EXP/'lcm_curated_final', alpha=8, rank=64)
print('Download MyDrive/ImagineAI_LCM/exports/* into the repo under adapters/')